# AI-Powered Industrial Component Identification: Preprocessing & EDA
**Mid-milestone notebook**

**Goal of this notebook:** prepare the image dataset for training the component classifier (Phase A of the architecture):

1. Load the dataset and build a metadata table
2. Data-quality checks (corrupt files, duplicates, tiny images)
3. Exploratory Data Analysis (class balance, image sizes, brightness / contrast / sharpness, colour, sample images)
4. Preprocessing pipeline (letterbox resize, normalisation) and augmentation
5. Stratified train / val / test split, class weights, and saved artefacts

**Expected folder layout** (either works, the notebook auto-detects it):

```
dataset/                          dataset/
  train/<class_name>/*.jpg          <class_name>/*.jpg
  valid/<class_name>/*.jpg
  test/<class_name>/*.jpg
```

If you downloaded a **detection** dataset in COCO format (e.g. from Roboflow), run the optional cropping cell in Section 1 first.

## 0. Setup

In [ ]:
import os, json, hashlib, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ------------------------- CONFIG -------------------------
DATA_DIR = Path("dataset")        # folder that holds the images (see layout above)
OUT_DIR  = Path("processed")      # where metadata / splits / weights are saved
IMG_SIZE = 224                    # input size for EfficientNet / MobileNet / ResNet
MIN_SIDE = 64                     # images with a shorter side than this are flagged as low-resolution
IMG_EXT  = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
# ----------------------------------------------------------
OUT_DIR.mkdir(exist_ok=True)
print("Dataset folder:", DATA_DIR.resolve(), "| exists:", DATA_DIR.exists())

## 1. Load the dataset

### 1a. (Optional) Convert a COCO detection export into cropped classification images
Some public datasets (e.g. Roboflow Universe) are annotated with bounding boxes. This helper crops every labelled component into
`dst_dir/<split>/<class_name>/` so it can be used for image classification. Skip it if your dataset is already in folder-per-class format.

In [ ]:
def coco_to_classification(src_dir, dst_dir, pad=0.05, min_size=16):
    """Crop each annotated object of a COCO export into class folders."""
    src_dir, dst_dir = Path(src_dir), Path(dst_dir)
    saved = 0
    for ann_file in src_dir.rglob("_annotations.coco.json"):
        split = ann_file.parent.name
        coco = json.load(open(ann_file))
        cats = {c["id"]: c["name"] for c in coco["categories"]}
        imgs = {i["id"]: i for i in coco["images"]}
        for a in coco["annotations"]:
            info = imgs[a["image_id"]]
            img = cv2.imread(str(ann_file.parent / info["file_name"]))
            if img is None:
                continue
            x, y, w, h = a["bbox"]
            px, py = int(w * pad), int(h * pad)
            x0, y0 = max(int(x) - px, 0), max(int(y) - py, 0)
            x1, y1 = min(int(x + w) + px, img.shape[1]), min(int(y + h) + py, img.shape[0])
            crop = img[y0:y1, x0:x1]
            if crop.size == 0 or min(crop.shape[:2]) < min_size:
                continue
            cls = cats[a["category_id"]].strip().replace(" ", "_")
            out = dst_dir / split / cls
            out.mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(out / f"{Path(info['file_name']).stem}_{a['id']}.jpg"), crop)
            saved += 1
    print(f"Saved {saved} cropped component images to '{dst_dir}'")

# Example (uncomment if you downloaded a COCO-format detection dataset into 'roboflow_download'):
# coco_to_classification("roboflow_download", "dataset")

### 1b. Build the metadata table

In [ ]:
SPLIT_NAMES = {"train": "train", "valid": "val", "val": "val", "validation": "val", "test": "test"}

def scan_dataset(root: Path):
    top_dirs = [d.name.lower() for d in root.iterdir() if d.is_dir()]
    has_splits = any(d in SPLIT_NAMES for d in top_dirs)
    rows = []
    for p in sorted(root.rglob("*")):
        if p.suffix.lower() not in IMG_EXT:
            continue
        rel = p.relative_to(root).parts
        if has_splits:
            split = SPLIT_NAMES.get(rel[0].lower(), "train")
            label = rel[1] if len(rel) > 2 else "unknown"
        else:
            split = None
            label = rel[0] if len(rel) > 1 else "unknown"
        rows.append({"path": str(p), "label": label, "orig_split": split})
    return pd.DataFrame(rows), has_splits

assert DATA_DIR.exists(), (f"Dataset folder '{DATA_DIR}' not found. Download the dataset and place it there "
                           "(if it is a COCO detection export, run the cropping cell in Section 1a first).")
df, has_splits = scan_dataset(DATA_DIR)
assert len(df) > 0, "No images found. Check DATA_DIR and the folder layout."
print(f"Images found : {len(df)}")
print(f"Classes      : {df['label'].nunique()}  ->  {sorted(df['label'].unique())}")
print(f"Pre-split data: {has_splits}")
df.head()

## 2. Data-quality checks
For every image we record size, brightness, contrast, sharpness (variance of the Laplacian) and mean colour.
We also detect **corrupt files** and **exact duplicates** (MD5 hash). Duplicates matter because the same image appearing in both
train and test would inflate accuracy (data leakage).

In [ ]:
def image_stats(path):
    try:
        img = cv2.imread(path)
        if img is None:
            return {"corrupt": True}
        h, w = img.shape[:2]
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        b, g, r = img.reshape(-1, 3).mean(axis=0)
        return {
            "corrupt": False, "width": w, "height": h, "aspect": w / h,
            "brightness": gray.mean(), "contrast": gray.std(),
            "sharpness": cv2.Laplacian(gray, cv2.CV_64F).var(),
            "mean_r": r, "mean_g": g, "mean_b": b,
            "filesize_kb": os.path.getsize(path) / 1024,
            "md5": hashlib.md5(Path(path).read_bytes()).hexdigest(),
        }
    except Exception:
        return {"corrupt": True}

stats = pd.DataFrame([image_stats(p) for p in df["path"]])
df = pd.concat([df.reset_index(drop=True), stats], axis=1)

n_corrupt = int(df["corrupt"].sum())
n_dupes   = int(df[~df["corrupt"]].duplicated("md5").sum())
n_lowres  = int(((df[["width", "height"]].min(axis=1)) < MIN_SIDE).sum())
print(f"Corrupt / unreadable images : {n_corrupt}")
print(f"Exact duplicate images      : {n_dupes}")
print(f"Low-resolution (< {MIN_SIDE}px side) : {n_lowres}")

In [ ]:
# Clean: drop corrupt files and exact duplicates (keep the first occurrence)
before = len(df)
df = df[~df["corrupt"]].drop_duplicates("md5").reset_index(drop=True)
df[["width", "height"]] = df[["width", "height"]].astype(int)
print(f"Removed {before - len(df)} images  ->  {len(df)} clean images remain")

## 3. Exploratory Data Analysis

### 3.1 Class distribution

In [ ]:
counts = df["label"].value_counts()
fig, ax = plt.subplots(figsize=(8, max(3, 0.45 * len(counts))))
sns.barplot(x=counts.values, y=counts.index, ax=ax, palette="viridis")
for i, v in enumerate(counts.values):
    ax.text(v + counts.max() * 0.01, i, str(v), va="center")
ax.set_title("Images per class"); ax.set_xlabel("Number of images"); ax.set_ylabel("")
plt.tight_layout(); plt.show()

imbalance = counts.max() / counts.min()
print(f"Imbalance ratio (largest / smallest class): {imbalance:.2f}")
print("Smallest classes:", {k: int(v) for k, v in counts.tail(3).items()})
if has_splits:
    display(pd.crosstab(df["label"], df["orig_split"]))

### 3.2 Image size and aspect ratio

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
sns.scatterplot(data=df, x="width", y="height", hue="label", s=18, alpha=0.7, ax=axes[0], legend=False)
axes[0].set_title("Width vs height (px)")
sns.histplot(df["aspect"], bins=30, ax=axes[1], color="#2b6cb0")
axes[1].set_title("Aspect ratio (w / h)")
sns.histplot(df["filesize_kb"], bins=30, ax=axes[2], color="#2f855a")
axes[2].set_title("File size (KB)")
plt.tight_layout(); plt.show()

display(df[["width", "height", "aspect", "filesize_kb"]].describe().round(2))
print("Unique image sizes:", df[["width", "height"]].drop_duplicates().shape[0])

### 3.3 Image quality per class: brightness, contrast, sharpness
Large differences between classes can let the model "cheat" by learning the lighting or background instead of the component itself.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, col, title in zip(axes, ["brightness", "contrast", "sharpness"],
                          ["Brightness (mean gray)", "Contrast (std gray)", "Sharpness (Laplacian variance)"]):
    sns.boxplot(data=df, x="label", y=col, ax=ax, palette="Set2")
    ax.set_title(title); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=45)
    if col == "sharpness":
        ax.set_yscale("log")
plt.tight_layout(); plt.show()

blurry = df[df["sharpness"] < df["sharpness"].quantile(0.05)]
print(f"Bottom 5% sharpest images (possible blur): {len(blurry)}")

### 3.4 Colour analysis
Colour is a useful cue for **material** (steel, brass, black-oxide, plastic), so we check it is not too inconsistent within a class.

In [ ]:
color = df.groupby("label")[["mean_r", "mean_g", "mean_b"]].mean().round(1)
display(color)

fig, ax = plt.subplots(figsize=(8, 3 + 0.2 * len(color)))
color.plot(kind="bar", ax=ax, color=["#e53e3e", "#38a169", "#3182ce"], width=0.8)
ax.set_title("Average RGB per class"); ax.set_xlabel(""); ax.set_ylabel("Mean pixel value")
ax.legend(["R", "G", "B"]); plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.show()

### 3.5 Sample images and the average image per class

In [ ]:
def read_rgb(path):
    return cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)

classes = sorted(df["label"].unique())
n_show = 5
fig, axes = plt.subplots(len(classes), n_show, figsize=(2.4 * n_show, 2.4 * len(classes)))
axes = np.atleast_2d(axes)
for i, c in enumerate(classes):
    sample = df[df["label"] == c].sample(min(n_show, (df["label"] == c).sum()), random_state=SEED)
    for j in range(n_show):
        ax = axes[i, j]; ax.axis("off")
        if j < len(sample):
            ax.imshow(read_rgb(sample.iloc[j]["path"]))
        if j == 0:
            ax.set_title(c, loc="left", fontsize=10, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Average image per class (resized to 128x128): shows the typical shape / position / background of each class
fig, axes = plt.subplots(1, len(classes), figsize=(2.3 * len(classes), 2.6))
axes = np.atleast_1d(axes)
for ax, c in zip(axes, classes):
    paths = df[df["label"] == c]["path"].sample(min(60, (df["label"] == c).sum()), random_state=SEED)
    avg = np.mean([cv2.resize(read_rgb(p), (128, 128)) for p in paths], axis=0).astype(np.uint8)
    ax.imshow(avg); ax.set_title(c, fontsize=9); ax.axis("off")
plt.suptitle("Average image per class", y=1.02); plt.tight_layout(); plt.show()

## 4. Preprocessing pipeline

Design choices (and why):

| Step | Choice | Reason |
|---|---|---|
| Resize | **Letterbox** (keep aspect ratio, pad to square) to 224x224 | Plain resizing would stretch long bolts and distort the shape/length, which is the cue that separates variants |
| Colour | Keep RGB, no hue jitter | Colour carries material information (steel vs brass vs black oxide) |
| Normalisation | ImageNet mean / std | Required for the pre-trained backbone (transfer learning) |

In [ ]:
def letterbox(img, size=IMG_SIZE, pad_value=114):
    """Resize keeping aspect ratio, then pad to a size x size square."""
    h, w = img.shape[:2]
    scale = size / max(h, w)
    nh, nw = max(1, int(round(h * scale))), max(1, int(round(w * scale)))
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR)
    canvas = np.full((size, size, 3), pad_value, dtype=np.uint8)
    top, left = (size - nh) // 2, (size - nw) // 2
    canvas[top:top + nh, left:left + nw] = resized
    return canvas

def preprocess(path, size=IMG_SIZE):
    """path -> float32 array (size, size, 3), ImageNet-normalised."""
    img = letterbox(read_rgb(path), size)
    return (img.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD

def denormalize(arr):
    return np.clip((arr * IMAGENET_STD + IMAGENET_MEAN), 0, 1)

# visual check: original -> letterboxed -> normalised tensor
ex = df.sample(4, random_state=SEED)
fig, axes = plt.subplots(3, 4, figsize=(11, 8.5))
for j, (_, r) in enumerate(ex.iterrows()):
    raw = read_rgb(r["path"]); proc = preprocess(r["path"])
    axes[0, j].imshow(raw);                axes[0, j].set_title(f"{r['label']}\noriginal {raw.shape[1]}x{raw.shape[0]}", fontsize=9)
    axes[1, j].imshow(letterbox(raw));     axes[1, j].set_title(f"letterbox {IMG_SIZE}x{IMG_SIZE}", fontsize=9)
    axes[2, j].imshow(denormalize(proc));  axes[2, j].set_title("after normalise (denorm. for display)", fontsize=8)
for ax in axes.ravel(): ax.axis("off")
plt.tight_layout(); plt.show()

t = preprocess(ex.iloc[0]["path"])
print("Tensor shape:", t.shape, "| dtype:", t.dtype, "| mean:", t.mean().round(3), "| std:", t.std().round(3))

### 4.1 Data augmentation
Warehouse photos will differ in angle, lighting and distance, so we simulate that during training. We avoid anything that would destroy the signal:
no hue shifts (material colour matters) and no aggressive cropping (the length of a part matters).

In [ ]:
def augment(img, rng):
    """Random rotation (parts can lie at any angle), flip, scale/shift, brightness/contrast, blur, noise."""
    h, w = img.shape[:2]
    angle = rng.uniform(-180, 180)
    scale = rng.uniform(0.85, 1.10)
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
    M[:, 2] += rng.uniform(-0.06, 0.06, size=2) * [w, h]
    out = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_CONSTANT, borderValue=(114, 114, 114))
    if rng.random() < 0.5:
        out = cv2.flip(out, 1)
    alpha, beta = rng.uniform(0.75, 1.25), rng.uniform(-25, 25)          # contrast, brightness
    out = np.clip(out.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
    if rng.random() < 0.3:
        k = int(rng.choice([3, 5]))
        out = cv2.GaussianBlur(out, (k, k), 0)
    if rng.random() < 0.3:
        out = np.clip(out + rng.normal(0, 6, out.shape), 0, 255).astype(np.uint8)
    return out

rng = np.random.default_rng(SEED)
base = letterbox(read_rgb(df.sample(1, random_state=7).iloc[0]["path"]))
fig, axes = plt.subplots(2, 5, figsize=(13, 5.4))
axes[0, 0].imshow(base); axes[0, 0].set_title("original", fontsize=10)
for ax in axes.ravel()[1:]:
    ax.imshow(augment(base, rng)); ax.set_title("augmented", fontsize=10)
for ax in axes.ravel(): ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Train / validation / test split, label encoding and class weights
If the dataset already ships with splits we keep them; otherwise we create a **stratified 70 / 15 / 15** split so every class appears in every subset in the same proportion.

In [ ]:
classes = sorted(df["label"].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
df["label_idx"] = df["label"].map(class_to_idx)

if has_splits:
    df["split"] = df["orig_split"]
else:
    min_count = df["label"].value_counts().min()
    strat = df["label_idx"] if min_count >= 3 else None
    if strat is None:
        print("Warning: a class has fewer than 3 images, falling back to a non-stratified split.")
    train_df, temp_df = train_test_split(df, test_size=0.30, stratify=strat, random_state=SEED)
    strat_t = temp_df["label_idx"] if strat is not None else None
    val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=strat_t, random_state=SEED)
    df["split"] = "train"
    df.loc[val_df.index, "split"] = "val"
    df.loc[test_df.index, "split"] = "test"

# leakage check: an image (hash) must not appear in more than one split
leak = df.groupby("md5")["split"].nunique().gt(1).sum()
print(f"Images present in more than one split (leakage): {leak}")

summary = pd.crosstab(df["label"], df["split"])[[c for c in ["train", "val", "test"] if c in df["split"].unique()]]
summary["total"] = summary.sum(axis=1)
display(summary)

In [ ]:
# Class weights (to counter class imbalance during training), computed on the training split only
train_labels = df.loc[df["split"] == "train", "label_idx"].values
weights = compute_class_weight("balanced", classes=np.arange(len(classes)), y=train_labels)
class_weights = {classes[i]: round(float(w), 3) for i, w in enumerate(weights)}
print("Class weights:", class_weights)

fig, ax = plt.subplots(figsize=(8, 3.2))
sns.barplot(x=list(class_weights.keys()), y=list(class_weights.values()), ax=ax, palette="mako")
ax.axhline(1.0, color="gray", ls="--", lw=1); ax.set_title("Class weights (1.0 = balanced)")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

## 6. Save artefacts for the training stage

In [ ]:
cols = ["path", "label", "label_idx", "split", "width", "height", "brightness", "contrast", "sharpness"]
df[cols].to_csv(OUT_DIR / "metadata.csv", index=False)
json.dump(class_to_idx, open(OUT_DIR / "class_to_idx.json", "w"), indent=2)
json.dump(class_weights, open(OUT_DIR / "class_weights.json", "w"), indent=2)
print("Saved to", OUT_DIR.resolve())
for f in sorted(OUT_DIR.iterdir()):
    print(" -", f.name)

## 7. Key observations *(update these after running on your dataset)*

- **Dataset size:** `N` clean images across `K` classes (corrupt / duplicate images removed: `...`).
- **Class balance:** imbalance ratio `...`; smallest class is `...` -> class weights + augmentation will be used.
- **Image properties:** sizes vary between `...` and `...` px; letterbox resizing preserves component proportions.
- **Quality / bias risks:** `...` (e.g. brightness differs between classes, plain background -> model may struggle on real warehouse photos).
- **Visual similarity:** `...` classes look alike (e.g. hex bolt vs hex-washer screw) -> expected to be the main source of confusion.

## 8. Next steps (remaining milestones)
1. Transfer learning with **EfficientNet-B0** (baselines: MobileNetV3, ResNet50) using `processed/metadata.csv` and the augmentation above
2. Evaluation: accuracy, macro-F1, top-3 accuracy, confusion matrix, error analysis, confidence calibration
3. Inventory database (parts, stock, rack / shelf / bin, machine compatibility) with simulated data
4. Web app: image upload -> top-3 prediction with confidence -> inventory lookup, plus text search, QR fallback, low-stock alerts, 2D warehouse map